# Experiment 2 (D3b) — Table Extraction: row-then-columns vs single-call

For a wide table form (one PDF → many rows: `interventions`, `outcomes`), compares the production **row-then-columns** two-stage extractor against a **single-call** arm (ask the whole table in one prompt). Only the table field's extraction strategy changes.

The single-call output is a list of row dicts, so it is **exploded** into one CSV row per table row (the production wide-long shape) before scoring. The row-then-columns arm = the production extraction already in `full_studies/<corpus>/claude/`.

**Kernel:** pick the env with `dspy` (`/home/ubuntu/miniconda3/envs/topics`). Cell 1 verifies it.

> Edited on disk while open? **File → Reload Notebook from Disk.**

In [1]:
# Cell 1 — environment setup
import sys, os, asyncio
sys.path.insert(0, '/home/ubuntu/evistream/backend')
sys.path.insert(0, '/home/ubuntu/evistream')
os.environ.setdefault('USE_RUNTIME_BUILDERS', 'true')

print('python:', sys.executable)
try:
    import dspy; print('dspy:', dspy.__version__)
except ModuleNotFoundError:
    raise SystemExit('No dspy in this kernel — switch to the topics env kernel (it has dspy 3.0.3).')

from dotenv import load_dotenv
for p in ('/home/ubuntu/evistream/eval/.env', '/home/ubuntu/evistream/backend/.env'):
    if os.path.exists(p):
        load_dotenv(p, override=False)
print('ANTHROPIC_API_KEY set:', bool(os.getenv('ANTHROPIC_API_KEY')))

python: /home/ubuntu/miniconda3/envs/topics/bin/python
dspy: 3.0.3
ANTHROPIC_API_KEY set: True


In [2]:
# Cell 2 — imports + experiment config
import pandas as pd
from eval.ablation.run_perio_ablation import PERIO_FORMS, _load_base_schema_def, _load_papers
from eval.ablation.run_extraction import DEFAULT_CONCURRENCY
from eval.table_ablation.transform import (
    to_single_call, table_field_name, subform_cols, explode_table_results,
)
from eval.table_ablation.run_table_ablation import _make_config, _write_csv, OUT_DIR, TABLE_FORMS

# ---- knobs ----
FORM            = 'interventions'   # 'interventions' | 'outcomes'
PAPERS_LIMIT    = 2                 # None = all 29 (full run)
CONCURRENCY     = DEFAULT_CONCURRENCY
USE_LLM         = True              # LLM-as-judge for interpretive fields (scoring)
FORCE_REEXTRACT = False             # True = re-extract even if the CSV already exists
print('table forms:', TABLE_FORMS)
print(f'FORM={FORM}  PAPERS_LIMIT={PAPERS_LIMIT}  FORCE_REEXTRACT={FORCE_REEXTRACT}')

table forms: ('interventions', 'outcomes')
FORM=interventions  PAPERS_LIMIT=2  FORCE_REEXTRACT=False


## Step 1 — Inspect the single-call transform (free, no LLM)
Confirms the table field + its columns, and the `row_then_columns → single_call` flip.

In [3]:
# Cell 3 — show the table field, columns, and the strategy flip
base_def   = _load_base_schema_def(FORM)
field      = table_field_name(base_def)
cols       = subform_cols(base_def, field)
single_def = to_single_call(base_def)

print(f"BASE {base_def['schema_name']}")
print(f"  table field : {field}")
print(f"  columns ({len(cols)}): {cols}")
print(f"  row-then-columns (production / full_studies)  vs  single_call (this arm)")
print(f"  flipped to single_call: {single_def['table_ablation']['fields_flipped']}")

BASE dynamic_7d186a2f_InterventionCharacteristics
  table field : interventions
  columns (6): ['arm_label', 'comparison_summary', 'intervention_description', 'n_in_arm', 'duration_of_followup', 'cochrane_subgroup_category']
  row-then-columns (production / full_studies)  vs  single_call (this arm)
  flipped to single_call: ['interventions']


## Step 2 — Extract the single-call arm + explode to rows (LLM cost; cached)
Runs `build_pipeline().run_batch`, then explodes the list-of-rows into one CSV row per table row. Skips if the CSV exists unless `FORCE_REEXTRACT=True`.

In [4]:
# Cell 4 — extract single_call + explode (await directly; skips cached)
papers = _load_papers(PAPERS_LIMIT)
OUT_DIR.mkdir(parents=True, exist_ok=True)
csv_path = OUT_DIR / f'{FORM}_single_long.csv'
if csv_path.exists() and not FORCE_REEXTRACT:
    print(f"cached → {csv_path.name} (set FORCE_REEXTRACT=True to redo)")
else:
    cfg, v_def = _make_config(FORM, base_def)
    print(f"extracting single_call on {len(papers)} papers ...")
    pipeline = cfg.build_pipeline()
    sem = asyncio.Semaphore(CONCURRENCY)
    results = await pipeline.run_batch(papers, sem)
    rows = explode_table_results(results, papers, field, cols)
    _write_csv(rows, cols, csv_path)
print('CSV →', csv_path)

  Loaded 2 periodontitis papers from /home/ubuntu/evistream/eval/cache/markdown_perio
cached → interventions_single_long.csv (set FORCE_REEXTRACT=True to redo)
CSV → /home/ubuntu/evistream/eval/ai sheets/table/periodontitis/claude/interventions_single_long.csv


In [5]:
# Cell 5 — verify the explode: one row per arm, structured columns
df = pd.read_csv(OUT_DIR / f'{FORM}_single_long.csv')
print('total rows:', len(df), '| columns:', list(df.columns))
print('\nrows (arms) per paper:')
print(df['Paper'].value_counts())
display(df.head(8))

total rows: 6 | columns: ['Paper', 'paper', 'arm_label', 'comparison_summary', 'intervention_description', 'n_in_arm', 'duration_of_followup', 'cochrane_subgroup_category']

rows (arms) per paper:
Paper
Bukleta 2018    4
Artese 2015     2
Name: count, dtype: int64


,Paper,paper,arm_label,comparison_summary,intervention_description,n_in_arm,duration_of_followup,cochrane_subgroup_category
0,Artese 2015,Artese 2015,ST (n = 12),Supragingival therapy (ST) vs intensive period...,Supragingival scaling using an ultrasonic devi...,12,6 months,supragingival_only
1,Artese 2015,Artese 2015,IT (n = 12),Supragingival therapy (ST) vs intensive period...,Supra- and subgingival scaling and root planin...,12,6 months,SI_alone
2,Bukleta 2018,Bukleta 2018,T2DM - Tooth Extraction only,Tooth extraction alone vs tooth extraction + F...,Tooth extraction only (at least one tooth extr...,50,3 months,usual_care
3,Bukleta 2018,Bukleta 2018,T2DM - Tooth Extraction and FM-SRP,Tooth extraction alone vs tooth extraction + F...,Tooth extraction plus Full-Mouth Scaling and R...,50,3 months,SI_plus_mouthrinse
4,Bukleta 2018,Bukleta 2018,Non-Diabetic - Tooth Extraction only,Tooth extraction alone vs tooth extraction + F...,Tooth extraction only (at least one tooth extr...,30,3 months,usual_care
5,Bukleta 2018,Bukleta 2018,Non-Diabetic - Tooth Extraction and FM-SRP,Tooth extraction alone vs tooth extraction + F...,Tooth extraction plus Full-Mouth Scaling and R...,30,3 months,SI_plus_mouthrinse


## Step 3 — Score row-then-columns vs single-call
Both arms scored with the same per-arm harness (`run_form` level-2 sub-record matching) vs `gt sheets/periodontitis.xlsx`. Runs as a **subprocess** to dodge the `config`/`forms` import collision.

row-then-columns = `full_studies/periodontitis/claude/` (production); single-call = the exploded CSV above. For a fair Δ, run the full 29 papers (`PAPERS_LIMIT=None`).

In [ ]:
# Cell 6 — score in a fresh process, then read the metrics sheet
import subprocess
cmd = [sys.executable, '-m', 'eval.table_ablation.score_table_ablation', '--form', FORM]
if not USE_LLM:
    cmd.append('--no-llm')
print('running:', ' '.join(cmd))
res = subprocess.run(cmd, cwd='/home/ubuntu/evistream', capture_output=True, text=True)
print(res.stdout[-3800:])
if res.returncode != 0:
    print('--- STDERR ---\n', res.stderr[-2500:])

metrics_path = f'/home/ubuntu/evistream/eval/outputs/table/periodontitis/claude/{FORM}_metrics.xlsx'
display(pd.read_excel(metrics_path))

## Notes
- **Two table forms:** `interventions` and `outcomes` — run each.
- **row-then-columns arm** is the production extraction (`full_studies/`), not re-run here; the scorer reads it directly for the Δ.
- **Caching:** Cell 4 skips if the single CSV exists; delete it or set `FORCE_REEXTRACT=True` to redo.
- **Full run:** `PAPERS_LIMIT = None`, then re-run for both forms.
- **CLI equivalent:** `python -m eval.table_ablation.run_table_ablation --form interventions` then `... .score_table_ablation --form interventions`.
- The explode lives in `transform.explode_table_results`; the single-call flip in `transform.to_single_call`.